# FieldTools — Campos electricos en sitio activo de RuBisCO

Calcula el campo electrico que la enzima genera en el sitio activo para estabilizar el estado de transicion.

Concepto clave de AI.zymes: el campo electrico catalitico es una metrica cuantificable para la seleccion de Boltzmann.

**Pipeline:** PDB → pdb2pqr → APBS → calculo de campo en residuos del sitio activo

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from libreria_analisismolecular import ai_zymes

sns.set_style('whitegrid')
print('Imports OK')

## 1. Definir residuos del sitio activo de RuBisCO

In [ ]:
# Residuos clave del sitio activo de RuBisCO (Form I)
# Basado en literatura (Andersson 2004, Poudel 2020)
ACTIVE_SITE_RESIDUES = [
    'LYS166',   # Carbamilacion, union a CO2
    'LYS168',   # Interaccion con sustrato
    'LYS172',   # Estabilizacion
    'LYS175',   # Union a Mg2+
    'LYS177',   # Interaccion catalitica
    'ASP194',   # Coordinacion de Mg2+
    'GLU195',   # Coordinacion de Mg2+
    'LYS201',   # Carbamilacion
    'HIS287',   # Interaccion con sustrato
    'LYS329',   # Estabilizacion
    'LYS331',   # Interaccion catalitica
    'LYS334',   # Union a Mg2+
]

print(f'Residuos del sitio activo: {len(ACTIVE_SITE_RESIDUES)}')
for res in ACTIVE_SITE_RESIDUES:
    print(f'  {res}')

## 2. Calcular campo electrico para cada PDB

In [ ]:
PDB_DIR = Path('/content/pdb/chain_a')
EFIELD_DIR = Path('/content/efield_results')
EFIELD_DIR.mkdir(exist_ok=True)

efield_results = []

for pdb_file in sorted(PDB_DIR.glob('*.pdb')):
    pid = pdb_file.stem.replace('_A', '')
    print(f'\nCalculando campo electrico para {pid}...')
    
    try:
        result = ai_zymes.calculate_electric_field(
            str(pdb_file),
            site_residues=ACTIVE_SITE_RESIDUES,
            output_dir=str(EFIELD_DIR / pid),
        )
        
        efield_results.append(result)
        print(f'  -> Magnitud del campo: {result["field_magnitude"]:.4f} V/A')
        
    except Exception as e:
        print(f'  -> Error: {e}')

df_efield = pd.DataFrame(efield_results)
if not df_efield.empty:
    display(df_efield)

## 3. Correlacion campo electrico vs especificidad (S)

In [ ]:
# Agregar valores de S (especificidad CO2/O2)
S_VALUES = {
    '4RUB': 77,
    '1GK8': 61,
    '1RBL': 13,
    '1GEH': 15,
    '2RUS': 5,
}

if not df_efield.empty:
    df_efield['S'] = df_efield['pdb'].map(S_VALUES)
    
    plt.figure(figsize=(8, 5))
    sns.scatterplot(
        data=df_efield, x='field_magnitude', y='S',
        s=150, hue='pdb', palette='Set1'
    )
    plt.xlabel('Magnitud del campo electrico (V/A)')
    plt.ylabel('Especificidad CO2/O2 (S)')
    plt.title('Campo electrico vs Especificidad en RuBisCO')
    plt.tight_layout()
    plt.show()
    
    # Correlacion
    corr = df_efield['field_magnitude'].corr(df_efield['S'])
    print(f'Correlacion campo electrico - S: {corr:.3f}')
    print(f'\nInterpretacion:')
    if corr > 0.5:
        print('  Correlacion positiva: mayor campo electrico -> mayor especificidad')
    elif corr < -0.5:
        print('  Correlacion negativa: menor campo electrico -> mayor especificidad')
    else:
        print('  Correlacion debil o nula')

## 4. Usar campo electrico como metrica para seleccion de Boltzmann

In [ ]:
# El campo electrico es una de las 3 metricas de AI.zymes:
# 1. Afinidad al estado de transicion
# 2. Estabilidad termodinamica
# 3. Campo electrico en sitio activo

if not df_efield.empty:
    # Normalizar campo electrico para usar como score
    df_efield['efield_score'] = (
        df_efield['field_magnitude'] - df_efield['field_magnitude'].min()
    ) / (
        df_efield['field_magnitude'].max() - df_efield['field_magnitude'].min()
    )
    
    print('Scores de campo electrico (normalizados 0-1):')
    for _, row in df_efield.iterrows():
        print(f'  {row["pdb"]}: {row["efield_score"]:.3f}')
    
    print(f'\nMejor campo electrico: {df_efield.loc[df_efield["efield_score"].idxmax(), "pdb"]}')
    print('\nProximo paso: combinar con afinidad y estabilidad en seleccion de Boltzmann')